# 🚀 Remotion 3D GPU Cloud Worker (NVIDIA Tesla T4)
### ⚡ Полный автопилот: Рендеринг видео по API с автоматическим отключением сессии

**Что делает этот блокнот:**
1. Подключает бесплатный GPU **NVIDIA Tesla T4 (16 GB VRAM)**.
2. Поднимает асинхронный сервер рендеринга (FastAPI + Cloudflare Tunnel).
3. Принимает задачи рендера из чата по API, рендерит за 20–30 секунд и отдает готовый MP4.
4. **Защита квот:** Сам выключает сессию Colab через 15 минут бездействия или по сигналу окончания рендера.


In [ ]:
# ==============================================================================
# 🚀 НАЖМИТЕ PLAY ДЛЯ ЗАПУСКА GPU СЕРВЕРА (ВСЕГО 1 КЛИК)
# ==============================================================================
import os, sys, subprocess, threading, time, json, shutil

# 1. Проверка GPU
gpu_res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_res.returncode != 0:
    print("⚠️ ВНИМАНИЕ: Среда без GPU! Выберите в меню: Среда выполнения -> Сменить тип -> T4 GPU")
else:
    print("⚡ NVIDIA Tesla T4 GPU обнаружена и готова!")

# 2. Установка системных зависимостей и Cloudflare
print("📦 Установка зависимостей (FastAPI, xvfb, cloudflared)...")
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "xvfb"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "python-multipart", "requests"], stdout=subprocess.DEVNULL)

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# 3. Настройка рабочей директории Remotion
BASE = "/content/render_service"
os.makedirs(BASE, exist_ok=True)
os.chdir(BASE)

pkg = {
    "name": "remotion_gpu_worker",
    "dependencies": {
        "@react-three/drei": "^9.122.0",
        "@react-three/fiber": "^8.18.0",
        "@remotion/cli": "^4.0.220",
        "@remotion/three": "^4.0.220",
        "react": "^18.2.0",
        "react-dom": "^18.2.0",
        "remotion": "^4.0.220",
        "three": "^0.160.0"
    }
}
with open(os.path.join(BASE, "package.json"), "w") as f:
    json.dump(pkg, f, indent=2)

if not os.path.exists(os.path.join(BASE, "node_modules")):
    print("⚙️ Первичная установка Remotion & Three.js (~35 сек)...
")
    subprocess.run(["npm", "install", "-s"], cwd=BASE)
    subprocess.run(["npx", "remotion", "browser", "ensure"], cwd=BASE)

# 4. Серверный код FastAPI
server_code = """import os, zipfile, subprocess, threading, time
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse

app = FastAPI(title="Remotion GPU Worker")
BASE = "/content/render_service"
STATUS = {"state": "idle", "composition": None, "file": None, "error": None, "log": "", "updated": time.time()}
LAST_ACTIVITY = time.time()

def watchdog_auto_shutdown():
    global LAST_ACTIVITY
    while True:
        time.sleep(30)
        if STATUS["state"] == "idle" and (time.time() - LAST_ACTIVITY > 900):
            print("⏱️ Бездействие более 15 минут. Выключаем Colab для сохранения квот...")
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception:
                pass
            break

threading.Thread(target=watchdog_auto_shutdown, daemon=True).start()

def execute_render(composition: str, zip_path: str):
    global STATUS, LAST_ACTIVITY
    LAST_ACTIVITY = time.time()
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(BASE)
        out_mp4 = os.path.join(BASE, f"{composition}.mp4")
        if os.path.exists(out_mp4):
            os.remove(out_mp4)
        STATUS["state"] = "rendering"
        STATUS["composition"] = composition
        STATUS["updated"] = time.time()
        runner = os.path.join(BASE, "run_render.py")
        if os.path.exists(runner):
            cmd = [sys.executable, runner, composition, out_mp4]
        else:
            cmd = ["xvfb-run", "-a", "npx", "remotion", "render", "src/index.ts", composition, out_mp4, "--concurrency=2", "--gl=angle-egl"]
        res = subprocess.run(cmd, cwd=BASE, capture_output=True, text=True)
        STATUS["log"] = res.stdout[-1500:] if res.stdout else ""
        if res.returncode != 0:
            STATUS["state"] = "error"
            STATUS["error"] = res.stderr[-2000:] or res.stdout[-2000:]
        else:
            STATUS["state"] = "done"
            STATUS["file"] = out_mp4
    except Exception as e:
        STATUS["state"] = "error"
        STATUS["error"] = str(e)
    finally:
        LAST_ACTIVITY = time.time()

@app.post("/render")
async def render(composition: str = Form("DrillProgression"), bundle: UploadFile = File(...)):
    global STATUS, LAST_ACTIVITY
    LAST_ACTIVITY = time.time()
    zip_path = os.path.join(BASE, "current_bundle.zip")
    with open(zip_path, "wb") as f:
        f.write(await bundle.read())
    STATUS = {"state": "starting", "composition": composition, "file": None, "error": None, "log": "Запуск...", "updated": time.time()}
    threading.Thread(target=execute_render, args=(composition, zip_path), daemon=True).start()
    return {"status": "started", "composition": composition}

@app.get("/status")
def get_status():
    global LAST_ACTIVITY
    LAST_ACTIVITY = time.time()
    return STATUS

@app.get("/download")
def download():
    global LAST_ACTIVITY
    LAST_ACTIVITY = time.time()
    if STATUS["state"] == "done" and STATUS["file"] and os.path.exists(STATUS["file"]):
        return FileResponse(STATUS["file"], media_type="video/mp4", filename=f"{STATUS.get('composition', 'video')}.mp4")
    return JSONResponse(status_code=400, content={"error": "Not ready", "status": STATUS})

@app.post("/shutdown")
def shutdown():
    def do_shutdown():
        time.sleep(1)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            pass
    threading.Thread(target=do_shutdown, daemon=True).start()
    return {"status": "Colab runtime is unassigning, quotas saved!"}

@app.get("/ping")
def ping():
    return {"status": "GPU Tesla T4 Online", "time": time.time()}
"""
with open(os.path.join(BASE, "server.py"), "w") as f:
    f.write(server_code)

# 5. Перезапуск uvicorn в daemon thread
subprocess.run(["fuser", "-k", "8000/tcp"])
subprocess.run(["pkill", "-f", "uvicorn"])
subprocess.run(["pkill", "-f", "cloudflared"])
time.sleep(1)

import uvicorn, requests
server_thread = threading.Thread(target=lambda: uvicorn.run("server:app", host="127.0.0.1", port=8000, log_level="warning"), daemon=True)
server_thread.start()

for i in range(10):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/ping", timeout=2)
        if r.status_code == 200:
            print("✅ FastAPI сервер успешно запущен!")
            break
    except Exception:
        pass
else:
    print("❌ Сервер не ответил.")

# 6. Запуск туннеля Cloudflare
tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in tunnel_proc.stdout:
    if "trycloudflare.com" in line:
        for part in line.split():
            if "trycloudflare.com" in part and part.startswith("http"):
                public_url = part
                try:
                    requests.post("https://ntfy.sh/remotion_gpu_sync_98a7f14b62e0c", data=public_url, timeout=5)
                    print("⚡ [Zero-Touch] Ссылка АВТОМАТИЧЕСКИ синхронизирована с чатом! Копировать ничего не нужно.")
                except Exception:
                    pass
                break
        if public_url:
            break

print("
" + "=" * 75)
print(f"🔥 ВАШ БЕСПЛАТНЫЙ GPU API АДРЕС:
{public_url}")
print("=" * 75)
print("💡 Скопируйте ссылку и отправьте в чат — дальше всё будет происходить автоматически!")
print("💡 Если не будет задач в течение 15 минут, сессия сама отключится для сохранения квот.")

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("Остановлено.")
